# 03 · Waste Description Classification
**Owner: Jeff**

Part of the EcoSort Waste Management System group project.

## Environment setup (works in **VS Code** and **Google Colab**)
Run this cell first. It detects where you are, gets the code + dataset, and installs deps.
- **VS Code / local:** it uses the repo files already on disk. Put `realwaste.zip` in `data/` (or let it auto-download).
- **Colab:** it clones the repo and downloads the dataset straight to Colab's fast local disk (no Google Drive needed).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub *Release* asset URL for realwaste.zip here

if IN_COLAB:
    # 1. get the code
    if not os.path.exists("Ecosort-Waste-Management-System"):
        !git clone https://github.com/REPLACE_ME/Ecosort-Waste-Management-System.git
    %cd Ecosort-Waste-Management-System
    # 2. install deps
    !pip -q install -r requirements.txt
    # 3. get the dataset onto Colab local disk (fast) — not Drive
    if not os.path.exists("data/RealWaste"):
        if DATASET_URL:
            !wget -q -O data/realwaste.zip "$DATASET_URL"
        else:
            print("Set DATASET_URL (GitHub Release asset) or upload data/realwaste.zip")

# make `src` importable from the repo root in both environments
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT); os.chdir(ROOT)
sys.path.insert(0, ROOT)

from src import config as C, data_prep
print("Ready. Categories:", C.CATEGORIES)


## Part 3 — Text classifier
Option A baseline (TF-IDF + LogReg). Add DistilBERT for the top tier.

In [ ]:
from src import text_classifier
train_df,val_df,test_df = data_prep.load_descriptions(split=True)
pipe = text_classifier.train(train_df,val_df)

In [ ]:
acc,cm,rep = text_classifier.evaluate(pipe,test_df)
print('Test accuracy:',acc)
print(rep)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.heatmap(cm,annot=True,fmt='d',xticklabels=C.CATEGORIES,yticklabels=C.CATEGORIES)
plt.xticks(rotation=45,ha='right'); plt.title('Text confusion matrix'); plt.tight_layout(); plt.show()

In [ ]:
text_classifier.predict_text('greasy pizza box with cheese stuck on it')

### TODO (Jeff)
- Inspect misclassifications & confusion cases
- OPTIONAL Option B: fine-tune DistilBERT (all layers) and compare
- Justify architecture choice